# train3: temporary training notebook (methods from outside NLP, part 2: data pruning and importance sampling)

First every text of the 200K mix is scored by how far a trained student still is from its teachers (`turkish_distill.score_texts`: 1 - cosine between the student's head output and the teacher vector, averaged over BGE-M3 and plain 8K-embed), using the finished plain pair at 10,800 steps (`scale_two_s10800_lr1e4`). Then two runs spend the same budget on different texts (data pruning, Sorscher et al. 2022; importance sampling, Katharopoulos & Fleuret 2018; both from vision):
1. `nn_prune_easy70_s5400`: the easiest 30% of the training texts are dropped, batches come from the hardest 70%.
2. `nn_weighted_s5400`: all texts, every batch drawn with probability proportional to the score.

Recipe as the reference (1k-base student, BGE-M3 + plain 8K-embed, lr 1e-4, 5400 steps, batch 128, seed 42); reference `scale_two_lr1e4_mix200k_s5400`, TR-MTEB 67.16. The scores file is written once to Drive (`bitirme/data/scores/`) and reused. Risk: the hardest texts may be noise; if pruning the easy ones helps, the opposite cut (`train.sampling=prune_hard`) is the control. Re-running the cells after a disconnect resumes the unfinished run and skips finished ones, then the last cell releases the runtime. Delete this notebook when the results are in.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# Runs of this notebook, one after another. (name, config, extra overrides). A later override wins.
CONFIG = 'configs/stage1_two_bge_m3_modernbert_tr_embed.yaml'   # BGE-M3 + plain 8K-embed
BASE = ["data.corpus_path=/content/drive/MyDrive/bitirme/data/mix/mix_v2_200k.jsonl", "train.max_steps=5400", "train.lr=0.0001"]
SCORES = f'{DRIVE_ROOT}/data/scores/two_plain_s10800_mix200k.npz'
SCORER = f'{DRIVE_ROOT}/runs/scale_two_s10800_lr1e4/20261027-152514-b2b0c4b'
if not os.path.exists(SCORES):
    os.makedirs(os.path.dirname(SCORES), exist_ok=True)
    get_ipython().system(f'python -m turkish_distill.score_texts --config {CONFIG} --model {SCORER} --out {SCORES} --batch-size 128 '
                         + ' '.join(f'--override {o}' for o in BASE))
SAMPLING = [f'train.sample_scores={SCORES}']
RUNS = [
    ('nn_prune_easy70_s5400', CONFIG, SAMPLING + ['train.sampling=prune_easy', 'train.keep_fraction=0.7']),
    ('nn_weighted_s5400', CONFIG, SAMPLING + ['train.sampling=weighted', 'train.sampling_power=1.0']),
]

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + BASE + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')

print('\nEval presets (paste into an eval notebook):')
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        run = os.path.basename(os.path.dirname(d))
        for suffix in ('', '_ema', '_swa'):
            if os.path.isdir(f'{DRIVE_ROOT}/runs/{name}/{run}/final{suffix}'):
                print(f"    'd_{name}{suffix}': dict(model=f'{{DRIVE_ROOT}}/runs/{name}/{run}/final{suffix}'),")


In [ ]:
from google.colab import runtime
runtime.unassign()